# HHVietSub — OmniVoice GPU Server
Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chọn **Runtime → Run all**.

In [ ]:
import pathlib, shutil
for folder in ('/content/HHVietSub-Colab', '/content/OmniVoice'):
    shutil.rmtree(folder, ignore_errors=True)
!nvidia-smi
!git clone --depth 1 https://github.com/nguyenduchung98/HHVietSub-Colab.git /content/HHVietSub-Colab
!git clone --depth 1 https://github.com/k2-fsa/OmniVoice.git /content/OmniVoice
!pip -q install -e /content/OmniVoice -r /content/HHVietSub-Colab/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print('Cài đặt hoàn tất.')

In [ ]:
import os, re, secrets, subprocess, threading, time, urllib.request
token = secrets.token_urlsafe(24)
env = {**os.environ, 'HHVIETSUB_TOKEN': token}
api = subprocess.Popen(
    ['uvicorn', 'server:app', '--host', '127.0.0.1', '--port', '3920'],
    cwd='/content/HHVietSub-Colab', env=env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1)
api_log = []
def read_api_log():
    for line in api.stdout:
        api_log.append(line.rstrip())
        print('[API]', line.rstrip())
threading.Thread(target=read_api_log, daemon=True).start()
health_request = urllib.request.Request(
    'http://127.0.0.1:3920/health', headers={'Authorization': f'Bearer {token}'})
deadline = time.time() + 120
while time.time() < deadline:
    if api.poll() is not None:
        raise RuntimeError('Uvicorn đã dừng. Xem các dòng [API] phía trên để biết lỗi.')
    try:
        with urllib.request.urlopen(health_request, timeout=3) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    api.terminate()
    raise TimeoutError('API không sẵn sàng sau 120 giây. Xem log [API] phía trên.')
print('API đã chạy. Đang nạp model OmniVoice vào GPU (lần đầu có thể mất vài phút)...')
warmup_request = urllib.request.Request(
    'http://127.0.0.1:3920/warmup', method='POST',
    headers={'Authorization': f'Bearer {token}'})
with urllib.request.urlopen(warmup_request, timeout=900) as response:
    if response.status != 200:
        raise RuntimeError('Không thể nạp model OmniVoice.')
print('Model OmniVoice đã nạp xong. Đang mở tunnel...')
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3920'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('Cloudflare không tạo được URL tunnel.')
print('\nURL HHVIETSUB:', public_url)
print('TOKEN BẢO MẬT:', token)
print('Dán cả URL và TOKEN vào HHVietSub, sau đó bấm Kiểm tra kết nối.')
threading.Thread(target=lambda: [print('[TUNNEL]', x.rstrip()) for x in tunnel.stdout], daemon=True).start()